In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
from importlib.metadata import version
from transformers import RobertaTokenizer

CURRENT_WORKING_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    path
    for path in (
        CURRENT_WORKING_DIR,
        *CURRENT_WORKING_DIR.parents,
    )
    if (path / "data" / "processed_rework_v2").exists()
)

CANDIDATE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
    / "01_candidate_population_index.sqlite"
)

CODEBERT_CHECKPOINT = "microsoft/codebert-base"
CODEBERT_MAX_LENGTH = 512
LANGUAGES = (
    "go",
    "java",
    "javascript",
    "php",
    "python",
    "ruby",
)

assert CANDIDATE_DB_PATH.exists(), (
    f"Candidate database not found: {CANDIDATE_DB_PATH}"
)

database_uri = (
    f"file:{CANDIDATE_DB_PATH.as_posix()}?mode=ro"
)

connection = sqlite3.connect(
    database_uri,
    uri=True,
    timeout=120,
)

try:
    database_metadata = pd.read_sql_query("""
        SELECT key, value
        FROM build_metadata
        ORDER BY key
    """, connection)

    population_summary = pd.read_sql_query("""
        SELECT
            language,
            original_split,
            COUNT(*) AS records,
            COUNT(DISTINCT repository) AS repositories
        FROM candidate_records
        GROUP BY language, original_split
        ORDER BY language, original_split
    """, connection)

    indexed_record_count = connection.execute(
        "SELECT COUNT(*) FROM candidate_records"
    ).fetchone()[0]

finally:
    connection.close()

codebert_tokenizer = RobertaTokenizer.from_pretrained(
    CODEBERT_CHECKPOINT,
    local_files_only=True,
)

pair_special_token_count = (
    codebert_tokenizer.num_special_tokens_to_add(
        pair=True
    )
)

print("Environment configuration")
print("-" * 90)
print(f"Project root          : {PROJECT_ROOT}")
print(f"Candidate database    : {CANDIDATE_DB_PATH}")
print(f"Indexed records       : {indexed_record_count:,}")
print(f"Transformers version  : {version('transformers')}")
print(f"Tokenizer             : {type(codebert_tokenizer).__name__}")
print(f"Model maximum length  : {codebert_tokenizer.model_max_length}")
print(f"Pair special tokens   : {pair_special_token_count}")

print("\nDatabase metadata")
display(database_metadata)

print("\nPopulation by language and partition")
display(population_summary)

assert indexed_record_count == 2_070_536
assert codebert_tokenizer.model_max_length == CODEBERT_MAX_LENGTH
assert pair_special_token_count == 4
assert set(population_summary["language"]) == set(LANGUAGES)

print("\nELIGIBILITY ENVIRONMENT READY")

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Environment configuration
------------------------------------------------------------------------------------------
Project root          : C:\Users\HP\Desktop\thesis_preprocessing
Candidate database    : C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\01_candidate_population_index.sqlite
Indexed records       : 2,070,536
Transformers version  : 4.56.2
Tokenizer             : RobertaTokenizer
Model maximum length  : 512
Pair special tokens   : 4

Database metadata


,key,value
0,build_status,complete
1,completed_at_utc,2026-08-05T07:34:11.360163+00:00
2,languages,"go,java,javascript,php,python,ruby"
3,raw_record_count_expected,2070536
4,schema_version,1
5,source_dataset,CodeSearchNet



Population by language and partition


,language,original_split,records,repositories
0,go,test,14291,213
1,go,train,317832,3821
2,go,valid,14242,212
3,java,test,26909,239
4,java,train,454451,4292
5,java,valid,15328,238
6,javascript,test,6483,882
7,javascript,train,123889,15858
8,javascript,valid,8253,881
9,php,test,28391,1069



ELIGIBILITY ENVIRONMENT READY


In [2]:
import gzip
import hashlib
import json
import time
from datetime import datetime, timezone
from transformers import RobertaTokenizerFast

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
TOKEN_LENGTH_CAP = CODEBERT_MAX_LENGTH + 1
TOKENIZATION_BATCH_SIZE = 256

try:
    audit_tokenizer = RobertaTokenizerFast.from_pretrained(
        CODEBERT_CHECKPOINT,
        local_files_only=True,
    )
except Exception:
    audit_tokenizer = codebert_tokenizer

probe_code = "def add(left, right): return left + right"
probe_comment = "Return the sum of two values."

slow_probe_ids = codebert_tokenizer(
    probe_code,
    probe_comment,
    add_special_tokens=True,
    truncation=False,
)["input_ids"]

fast_probe_ids = audit_tokenizer(
    probe_code,
    probe_comment,
    add_special_tokens=True,
    truncation=False,
)["input_ids"]

assert slow_probe_ids == fast_probe_ids

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    connection.execute("PRAGMA journal_mode = WAL")
    connection.execute("PRAGMA synchronous = NORMAL")
    connection.execute("PRAGMA temp_store = MEMORY")

    connection.execute("""
        CREATE TABLE IF NOT EXISTS codebert_length_audit (
            family_id TEXT PRIMARY KEY,
            pair_token_count_capped INTEGER NOT NULL,
            fits_512 INTEGER NOT NULL
                CHECK (fits_512 IN (0, 1))
        )
    """)

    connection.execute("""
        CREATE TABLE IF NOT EXISTS codebert_length_processed_shards (
            shard_relative_path TEXT PRIMARY KEY,
            language TEXT NOT NULL,
            original_split TEXT NOT NULL,
            source_size_bytes INTEGER NOT NULL,
            source_mtime_ns INTEGER NOT NULL,
            processed_records INTEGER NOT NULL,
            processed_at_utc TEXT NOT NULL
        )
    """)

    connection.execute("""
        CREATE TABLE IF NOT EXISTS codebert_length_metadata (
            key TEXT PRIMARY KEY,
            value TEXT NOT NULL
        )
    """)

    connection.execute("""
        CREATE INDEX IF NOT EXISTS idx_codebert_fits_512
        ON codebert_length_audit(fits_512)
    """)

    metadata = {
        "checkpoint": CODEBERT_CHECKPOINT,
        "transformers_version": version("transformers"),
        "tokenizer_class": type(audit_tokenizer).__name__,
        "maximum_model_length": str(CODEBERT_MAX_LENGTH),
        "stored_length_cap": str(TOKEN_LENGTH_CAP),
        "build_status": "running",
    }

    connection.executemany(
        """
        INSERT INTO codebert_length_metadata(key, value)
        VALUES (?, ?)
        ON CONFLICT(key)
        DO UPDATE SET value = excluded.value
        """,
        metadata.items(),
    )

    connection.commit()

    processed_shard_rows = connection.execute("""
        SELECT
            shard_relative_path,
            source_size_bytes,
            source_mtime_ns,
            processed_records
        FROM codebert_length_processed_shards
    """).fetchall()

    processed_shards = {
        row[0]: {
            "source_size_bytes": row[1],
            "source_mtime_ns": row[2],
            "processed_records": row[3],
        }
        for row in processed_shard_rows
    }

    insert_sql = """
        INSERT INTO codebert_length_audit (
            family_id,
            pair_token_count_capped,
            fits_512
        )
        VALUES (?, ?, ?)
    """

    def insert_tokenized_batch(
        family_ids,
        codes,
        comments,
    ):
        encoded = audit_tokenizer(
            codes,
            comments,
            add_special_tokens=True,
            truncation=True,
            max_length=TOKEN_LENGTH_CAP,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            verbose=False,
        )

        lengths = [
            len(input_ids)
            for input_ids in encoded["input_ids"]
        ]

        rows = [
            (
                family_id,
                token_count,
                int(token_count <= CODEBERT_MAX_LENGTH),
            )
            for family_id, token_count in zip(
                family_ids,
                lengths,
            )
        ]

        connection.executemany(
            insert_sql,
            rows,
        )

        return len(rows)

    build_started = time.perf_counter()
    newly_processed_records = 0

    for language in LANGUAGES:
        language_root = (
            RAW_DATA_DIR
            / language
            / "final"
            / "jsonl"
        )

        for split in ("train", "valid", "test"):
            shard_paths = sorted(
                (language_root / split).glob("*.jsonl.gz")
            )

            for shard_path in shard_paths:
                shard_relative_path = (
                    shard_path
                    .relative_to(PROJECT_ROOT)
                    .as_posix()
                )

                shard_stat = shard_path.stat()

                if shard_relative_path in processed_shards:
                    previous = processed_shards[
                        shard_relative_path
                    ]

                    if (
                        previous["source_size_bytes"]
                        != shard_stat.st_size
                        or previous["source_mtime_ns"]
                        != shard_stat.st_mtime_ns
                    ):
                        raise RuntimeError(
                            f"Raw shard changed: {shard_relative_path}"
                        )

                    print(
                        f"Skipped {shard_relative_path} | "
                        f"records={previous['processed_records']:,}"
                    )
                    continue

                shard_started = time.perf_counter()
                shard_processed = 0

                batch_family_ids = []
                batch_codes = []
                batch_comments = []

                with connection:
                    with gzip.open(
                        shard_path,
                        mode="rt",
                        encoding="utf-8",
                        errors="strict",
                    ) as handle:

                        for line_number, raw_line in enumerate(
                            handle,
                            start=1,
                        ):
                            if not raw_line.strip():
                                continue

                            record = json.loads(raw_line)

                            family_identity = "\x1f".join([
                                language,
                                split,
                                shard_relative_path,
                                str(line_number),
                                record["sha"],
                                record["url"],
                            ])

                            family_id = hashlib.sha256(
                                family_identity.encode("utf-8")
                            ).hexdigest()

                            batch_family_ids.append(family_id)
                            batch_codes.append(record["code"])
                            batch_comments.append(
                                record["docstring"]
                            )

                            if (
                                len(batch_family_ids)
                                >= TOKENIZATION_BATCH_SIZE
                            ):
                                shard_processed += (
                                    insert_tokenized_batch(
                                        batch_family_ids,
                                        batch_codes,
                                        batch_comments,
                                    )
                                )

                                batch_family_ids.clear()
                                batch_codes.clear()
                                batch_comments.clear()

                        if batch_family_ids:
                            shard_processed += (
                                insert_tokenized_batch(
                                    batch_family_ids,
                                    batch_codes,
                                    batch_comments,
                                )
                            )

                    connection.execute(
                        """
                        INSERT INTO
                        codebert_length_processed_shards (
                            shard_relative_path,
                            language,
                            original_split,
                            source_size_bytes,
                            source_mtime_ns,
                            processed_records,
                            processed_at_utc
                        )
                        VALUES (?, ?, ?, ?, ?, ?, ?)
                        """,
                        (
                            shard_relative_path,
                            language,
                            split,
                            shard_stat.st_size,
                            shard_stat.st_mtime_ns,
                            shard_processed,
                            datetime.now(
                                timezone.utc
                            ).isoformat(),
                        ),
                    )

                newly_processed_records += shard_processed
                elapsed = time.perf_counter() - shard_started

                print(
                    f"Tokenized {shard_relative_path} | "
                    f"records={shard_processed:,} | "
                    f"seconds={elapsed:.1f}"
                )

    final_record_count = connection.execute("""
        SELECT COUNT(*)
        FROM codebert_length_audit
    """).fetchone()[0]

    final_shard_count = connection.execute("""
        SELECT COUNT(*)
        FROM codebert_length_processed_shards
    """).fetchone()[0]

    missing_candidate_records = connection.execute("""
        SELECT COUNT(*)
        FROM candidate_records AS candidate
        LEFT JOIN codebert_length_audit AS audit
            ON candidate.family_id = audit.family_id
        WHERE audit.family_id IS NULL
    """).fetchone()[0]

    unmatched_audit_records = connection.execute("""
        SELECT COUNT(*)
        FROM codebert_length_audit AS audit
        LEFT JOIN candidate_records AS candidate
            ON audit.family_id = candidate.family_id
        WHERE candidate.family_id IS NULL
    """).fetchone()[0]

    eligibility_summary = pd.read_sql_query("""
        SELECT
            candidate.language,
            candidate.original_split,
            COUNT(*) AS records,
            SUM(audit.fits_512) AS fits_512,
            COUNT(*) - SUM(audit.fits_512) AS exceeds_512,
            ROUND(
                100.0 * SUM(audit.fits_512) / COUNT(*),
                2
            ) AS fits_512_percent
        FROM candidate_records AS candidate
        INNER JOIN codebert_length_audit AS audit
            ON candidate.family_id = audit.family_id
        GROUP BY
            candidate.language,
            candidate.original_split
        ORDER BY
            candidate.language,
            candidate.original_split
    """, connection)

    if (
        final_record_count == indexed_record_count
        and final_shard_count == 78
        and missing_candidate_records == 0
        and unmatched_audit_records == 0
    ):
        connection.execute(
            """
            INSERT INTO codebert_length_metadata(key, value)
            VALUES ('build_status', 'complete')
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """
        )

        connection.execute(
            """
            INSERT INTO codebert_length_metadata(key, value)
            VALUES ('completed_at_utc', ?)
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            (
                datetime.now(
                    timezone.utc
                ).isoformat(),
            ),
        )

        connection.commit()

finally:
    connection.close()

total_elapsed = time.perf_counter() - build_started

print("\nCodeBERT length eligibility")
print("-" * 90)
print(f"Newly processed records : {newly_processed_records:,}")
print(f"Final audited records   : {final_record_count:,}")
print(f"Processed shards        : {final_shard_count:,}")
print(f"Missing candidate rows  : {missing_candidate_records:,}")
print(f"Unmatched audit rows    : {unmatched_audit_records:,}")
print(f"Elapsed seconds         : {total_elapsed:.1f}")

display(eligibility_summary)

assert final_record_count == indexed_record_count
assert final_shard_count == 78
assert missing_candidate_records == 0
assert unmatched_audit_records == 0

print("\nCODEBERT LENGTH ELIGIBILITY COMPLETED")

Tokenized data/raw/go/final/jsonl/train/go_train_0.jsonl.gz | records=30,000 | seconds=5.8
Tokenized data/raw/go/final/jsonl/train/go_train_1.jsonl.gz | records=30,000 | seconds=6.6
Tokenized data/raw/go/final/jsonl/train/go_train_10.jsonl.gz | records=17,832 | seconds=2.4
Tokenized data/raw/go/final/jsonl/train/go_train_2.jsonl.gz | records=30,000 | seconds=3.9
Tokenized data/raw/go/final/jsonl/train/go_train_3.jsonl.gz | records=30,000 | seconds=3.2
Tokenized data/raw/go/final/jsonl/train/go_train_4.jsonl.gz | records=30,000 | seconds=4.1
Tokenized data/raw/go/final/jsonl/train/go_train_5.jsonl.gz | records=30,000 | seconds=4.6
Tokenized data/raw/go/final/jsonl/train/go_train_6.jsonl.gz | records=30,000 | seconds=5.5
Tokenized data/raw/go/final/jsonl/train/go_train_7.jsonl.gz | records=30,000 | seconds=5.0
Tokenized data/raw/go/final/jsonl/train/go_train_8.jsonl.gz | records=30,000 | seconds=4.5
Tokenized data/raw/go/final/jsonl/train/go_train_9.jsonl.gz | records=30,000 | seconds=9.

,language,original_split,records,fits_512,exceeds_512,fits_512_percent
0,go,test,14291,13159,1132,92.08
1,go,train,317832,294467,23365,92.65
2,go,valid,14242,13589,653,95.41
3,java,test,26909,22412,4497,83.29
4,java,train,454451,376960,77491,82.95
5,java,valid,15328,13131,2197,85.67
6,javascript,test,6483,5059,1424,78.03
7,javascript,train,123889,95381,28508,76.99
8,javascript,valid,8253,6445,1808,78.09
9,php,test,28391,23251,5140,81.90



CODEBERT LENGTH ELIGIBILITY COMPLETED


In [3]:
connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    with connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS eligibility_metadata (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL
            )
        """)

        connection.execute(
            "DROP VIEW IF EXISTS eligible_candidate_records"
        )

        connection.execute("""
            CREATE VIEW eligible_candidate_records AS
            SELECT
                candidate.*,
                audit.pair_token_count_capped
            FROM candidate_records AS candidate
            INNER JOIN codebert_length_audit AS audit
                ON candidate.family_id = audit.family_id
            WHERE audit.fits_512 = 1
        """)

        eligibility_metadata = {
            "eligibility_version": "1",
            "codebert_checkpoint": CODEBERT_CHECKPOINT,
            "maximum_pair_tokens": str(CODEBERT_MAX_LENGTH),
            "codebert_truncation_allowed": "false",
            "minimum_code_tokens": "none",
            "minimum_comment_tokens": "none",
            "minimum_code_lines": "none",
            "minimum_comment_lines": "none",
            "boilerplate_filter_applied": "false",
            "exact_code_duplicates_removed": "not applicable; none found",
            "exact_pair_duplicates_removed": "not applicable; none found",
        }

        connection.executemany(
            """
            INSERT INTO eligibility_metadata(key, value)
            VALUES (?, ?)
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            eligibility_metadata.items(),
        )

    eligibility_by_partition = pd.read_sql_query("""
        SELECT
            raw.language,
            raw.original_split,
            COUNT(*) AS raw_records,
            SUM(audit.fits_512) AS eligible_records,
            COUNT(*) - SUM(audit.fits_512)
                AS excluded_for_length,
            ROUND(
                100.0 * SUM(audit.fits_512) / COUNT(*),
                2
            ) AS eligible_percent
        FROM candidate_records AS raw
        INNER JOIN codebert_length_audit AS audit
            ON raw.family_id = audit.family_id
        GROUP BY
            raw.language,
            raw.original_split
        ORDER BY
            raw.language,
            raw.original_split
    """, connection)

    eligible_test_repository_sizes = pd.read_sql_query("""
        SELECT
            language,
            repository,
            COUNT(*) AS eligible_methods
        FROM eligible_candidate_records
        WHERE original_split = 'test'
        GROUP BY
            language,
            repository
        ORDER BY
            language,
            repository
    """, connection)

    eligible_record_count = connection.execute("""
        SELECT COUNT(*)
        FROM eligible_candidate_records
    """).fetchone()[0]

finally:
    connection.close()


test_repository_summary = (
    eligible_test_repository_sizes
    .groupby("language")["eligible_methods"]
    .agg(
        eligible_test_records="sum",
        eligible_test_repositories="count",
        minimum_methods="min",
        median_methods="median",
        mean_methods="mean",
        maximum_methods="max",
    )
    .reset_index()
)

test_repository_summary["p90_methods"] = (
    eligible_test_repository_sizes
    .groupby("language")["eligible_methods"]
    .quantile(0.90)
    .values
)

test_repository_summary["p95_methods"] = (
    eligible_test_repository_sizes
    .groupby("language")["eligible_methods"]
    .quantile(0.95)
    .values
)

test_repository_summary[
    [
        "median_methods",
        "mean_methods",
        "p90_methods",
        "p95_methods",
    ]
] = (
    test_repository_summary[
        [
            "median_methods",
            "mean_methods",
            "p90_methods",
            "p95_methods",
        ]
    ].round(2)
)


print("Official eligible population by partition")
display(eligibility_by_partition)

print("\nEligible test repositories")
display(test_repository_summary)

print(f"\nTotal eligible records: {eligible_record_count:,}")

assert eligible_record_count == int(
    eligibility_by_partition["eligible_records"].sum()
)

assert set(test_repository_summary["language"]) == set(
    LANGUAGES
)

print("\nELIGIBLE POPULATION CREATED")
print("No sample records were selected.")

Official eligible population by partition


,language,original_split,raw_records,eligible_records,excluded_for_length,eligible_percent
0,go,test,14291,13159,1132,92.08
1,go,train,317832,294467,23365,92.65
2,go,valid,14242,13589,653,95.41
3,java,test,26909,22412,4497,83.29
4,java,train,454451,376960,77491,82.95
5,java,valid,15328,13131,2197,85.67
6,javascript,test,6483,5059,1424,78.03
7,javascript,train,123889,95381,28508,76.99
8,javascript,valid,8253,6445,1808,78.09
9,php,test,28391,23251,5140,81.90



Eligible test repositories


,language,eligible_test_records,eligible_test_repositories,minimum_methods,median_methods,mean_methods,maximum_methods,p90_methods,p95_methods
0,go,13159,211,1,10.0,62.36,2577,104.0,212.0
1,java,22412,234,1,15.0,95.78,2788,251.7,409.8
2,javascript,5059,790,1,2.0,6.40,246,14.0,21.0
3,php,23251,1029,1,7.0,22.60,1041,44.0,79.6
4,python,14180,633,1,7.0,22.40,1077,43.0,73.4
5,ruby,1990,307,1,3.0,6.48,163,12.0,18.7



Total eligible records: 1,656,163

ELIGIBLE POPULATION CREATED
No sample records were selected.


In [4]:
import math
import sqlite3
from statistics import NormalDist

CANDIDATE_SAMPLE_SIZES = (
    250,
    500,
    750,
    1_000,
    1_250,
    1_500,
    1_750,
    1_900,
    2_000,
)

POWER = 0.90
GENERAL_ALPHA = 0.05
RQ1_PLANNED_CONTRASTS = 2
RQ2_LANGUAGE_PAIR_COMPARISONS = 15
COMPLEXITY_GROUPS = 4

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    eligible_test_population = pd.read_sql_query("""
        SELECT
            language,
            COUNT(*) AS eligible_test_records,
            COUNT(DISTINCT repository)
                AS eligible_test_repositories
        FROM eligible_candidate_records
        WHERE original_split = 'test'
        GROUP BY language
        ORDER BY language
    """, connection)

    eligible_repository_sizes = pd.read_sql_query("""
        SELECT
            language,
            repository,
            COUNT(*) AS eligible_methods
        FROM eligible_candidate_records
        WHERE original_split = 'test'
        GROUP BY language, repository
        ORDER BY language, repository
    """, connection)

finally:
    connection.close()


def two_sided_critical_z(alpha):
    return NormalDist().inv_cdf(
        1 - alpha / 2
    )


def power_z(power):
    return NormalDist().inv_cdf(power)


def finite_population_margin(
    population_size,
    sample_size,
    alpha,
):
    if sample_size > population_size:
        return None

    finite_population_correction = math.sqrt(
        (population_size - sample_size)
        / (population_size - 1)
    )

    return (
        two_sided_critical_z(alpha)
        * math.sqrt(0.25 / sample_size)
        * finite_population_correction
    )


def paired_effect_mde(
    sample_size,
    alpha,
    power,
):
    return (
        two_sided_critical_z(alpha)
        + power_z(power)
    ) / math.sqrt(sample_size)


def independent_correlation_difference_mde(
    sample_size,
    alpha,
    power,
):
    fisher_z_difference = (
        two_sided_critical_z(alpha)
        + power_z(power)
    ) * math.sqrt(
        2 / (sample_size - 3)
    )

    return math.tanh(fisher_z_difference)


def minimum_repository_cap(
    repository_sizes,
    sample_size,
):
    total_available = sum(repository_sizes)

    if sample_size > total_available:
        return None

    lower = 1
    upper = max(repository_sizes)

    while lower < upper:
        middle = (lower + upper) // 2

        capacity = sum(
            min(size, middle)
            for size in repository_sizes
        )

        if capacity >= sample_size:
            upper = middle
        else:
            lower = middle + 1

    return lower


rq1_alpha = (
    GENERAL_ALPHA
    / RQ1_PLANNED_CONTRASTS
)

rq2_alpha = (
    GENERAL_ALPHA
    / RQ2_LANGUAGE_PAIR_COMPARISONS
)

repository_sizes_by_language = {
    language: group["eligible_methods"].tolist()
    for language, group
    in eligible_repository_sizes.groupby("language")
}

design_rows = []

for population_row in eligible_test_population.itertuples(
    index=False
):
    language = population_row.language
    population_size = int(
        population_row.eligible_test_records
    )
    repository_count = int(
        population_row.eligible_test_repositories
    )
    repository_sizes = (
        repository_sizes_by_language[language]
    )

    for sample_size in CANDIDATE_SAMPLE_SIZES:
        feasible = sample_size <= population_size

        repository_cap = (
            minimum_repository_cap(
                repository_sizes,
                sample_size,
            )
            if feasible
            else None
        )

        maximum_repositories_represented = (
            min(sample_size, repository_count)
            if feasible
            else None
        )

        design_rows.append({
            "language": language,
            "sample_size": sample_size,
            "eligible_test_records": population_size,
            "eligible_test_repositories": repository_count,
            "feasible": feasible,
            "sampling_fraction_percent": (
                round(
                    100
                    * sample_size
                    / population_size,
                    2,
                )
                if feasible
                else None
            ),
            "worst_case_margin_percent": (
                round(
                    100
                    * finite_population_margin(
                        population_size,
                        sample_size,
                        GENERAL_ALPHA,
                    ),
                    3,
                )
                if feasible
                else None
            ),
            "rq1_paired_effect_mde_90": (
                round(
                    paired_effect_mde(
                        sample_size,
                        rq1_alpha,
                        POWER,
                    ),
                    4,
                )
                if feasible
                else None
            ),
            "rq2_correlation_gap_mde_90": (
                round(
                    independent_correlation_difference_mde(
                        sample_size,
                        rq2_alpha,
                        POWER,
                    ),
                    4,
                )
                if feasible
                else None
            ),
            "minimum_per_complexity_quartile": (
                sample_size // COMPLEXITY_GROUPS
                if feasible
                else None
            ),
            "minimum_repository_cap_needed": (
                repository_cap
            ),
            "maximum_repositories_represented": (
                maximum_repositories_represented
            ),
            "maximum_repository_coverage_percent": (
                round(
                    100
                    * maximum_repositories_represented
                    / repository_count,
                    2,
                )
                if feasible
                else None
            ),
        })


sample_size_detail = pd.DataFrame(
    design_rows
)


summary_rows = []

for sample_size in CANDIDATE_SAMPLE_SIZES:
    candidate = sample_size_detail.loc[
        sample_size_detail["sample_size"]
        == sample_size
    ]

    all_languages_feasible = bool(
        candidate["feasible"].all()
    )

    feasible_candidate = candidate.loc[
        candidate["feasible"]
    ]

    summary_rows.append({
        "sample_size_per_language": sample_size,
        "balanced_total_sample": (
            sample_size * len(LANGUAGES)
            if all_languages_feasible
            else None
        ),
        "all_languages_feasible": (
            all_languages_feasible
        ),
        "largest_sampling_fraction_percent": (
            feasible_candidate[
                "sampling_fraction_percent"
            ].max()
        ),
        "worst_margin_percent": (
            feasible_candidate[
                "worst_case_margin_percent"
            ].max()
        ),
        "rq1_paired_effect_mde_90": (
            feasible_candidate[
                "rq1_paired_effect_mde_90"
            ].max()
        ),
        "rq2_correlation_gap_mde_90": (
            feasible_candidate[
                "rq2_correlation_gap_mde_90"
            ].max()
        ),
        "minimum_per_complexity_quartile": (
            sample_size // COMPLEXITY_GROUPS
            if all_languages_feasible
            else None
        ),
        "largest_repository_cap_needed": (
            feasible_candidate[
                "minimum_repository_cap_needed"
            ].max()
        ),
        "lowest_max_repository_coverage_percent": (
            feasible_candidate[
                "maximum_repository_coverage_percent"
            ].min()
        ),
    })


sample_size_comparison = pd.DataFrame(
    summary_rows
)


print("Multiple-rule sample-size comparison")
display(sample_size_comparison)

print(
    "\nMinimum repository cap required "
    "for each candidate size"
)

display(
    sample_size_detail.pivot(
        index="language",
        columns="sample_size",
        values="minimum_repository_cap_needed",
    )
)

print(
    "\nWorst-case 95% margin of error "
    "by language"
)

display(
    sample_size_detail.pivot(
        index="language",
        columns="sample_size",
        values="worst_case_margin_percent",
    )
)

print(
    "\nMaximum repository coverage "
    "when distinct repositories are prioritized"
)

display(
    sample_size_detail.pivot(
        index="language",
        columns="sample_size",
        values="maximum_repository_coverage_percent",
    )
)

assert len(sample_size_comparison) == len(
    CANDIDATE_SAMPLE_SIZES
)

assert len(sample_size_detail) == (
    len(LANGUAGES)
    * len(CANDIDATE_SAMPLE_SIZES)
)

print("\nSAMPLE-SIZE DESIGN COMPARISON COMPLETED")
print("No sample size or records were selected.")

Multiple-rule sample-size comparison


,sample_size_per_language,balanced_total_sample,all_languages_feasible,largest_sampling_fraction_percent,worst_margin_percent,rq1_paired_effect_mde_90,rq2_correlation_gap_mde_90,minimum_per_complexity_quartile,largest_repository_cap_needed,lowest_max_repository_coverage_percent
0,250,1500.0,True,12.56,6.165,0.2228,0.3622,62.0,2.0,24.30
1,500,3000.0,True,25.13,4.335,0.1576,0.2613,125.0,3.0,48.59
2,750,4500.0,True,37.69,3.520,0.1286,0.2148,187.0,5.0,72.89
3,1000,6000.0,True,50.25,3.032,0.1114,0.1866,250.0,6.0,97.18
4,1250,7500.0,True,62.81,2.696,0.0996,0.1673,312.0,9.0,100.00
5,1500,9000.0,True,75.38,2.447,0.0910,0.1529,375.0,17.0,100.00
6,1750,10500.0,True,87.94,2.253,0.0842,0.1417,437.0,36.0,100.00
7,1900,11400.0,True,95.48,2.154,0.0808,0.1361,475.0,80.0,100.00
8,2000,NaN,False,39.53,2.095,0.0788,0.1327,NaN,16.0,100.00



Minimum repository cap required for each candidate size


sample_size,250,500,750,1000,1250,1500,1750,1900,2000
language,,,,,,,,,
go,2.0,3.0,5.0,6.0,8.0,11.0,13.0,15.0,16.0
java,2.0,3.0,4.0,5.0,7.0,8.0,10.0,11.0,12.0
javascript,1.0,1.0,1.0,2.0,2.0,3.0,4.0,4.0,5.0
php,1.0,1.0,1.0,1.0,2.0,2.0,2.0,2.0,3.0
python,1.0,1.0,2.0,2.0,3.0,3.0,4.0,4.0,4.0
ruby,1.0,2.0,4.0,6.0,9.0,17.0,36.0,80.0,NaN



Worst-case 95% margin of error by language


sample_size,250,500,750,1000,1250,1500,1750,1900,2000
language,,,,,,,,,
go,6.139,4.299,3.475,2.979,2.637,2.382,2.181,2.080,2.018
java,6.163,4.334,3.518,3.029,2.693,2.444,2.249,2.151,2.091
javascript,6.043,4.161,3.303,2.776,2.405,2.122,1.895,1.777,1.704
php,6.165,4.335,3.520,3.032,2.696,2.447,2.253,2.154,2.095
python,6.143,4.305,3.483,2.988,2.647,2.393,2.193,2.092,2.031
ruby,5.797,3.793,2.825,2.186,1.691,1.256,0.814,0.478,NaN



Maximum repository coverage when distinct repositories are prioritized


sample_size,250,500,750,1000,1250,1500,1750,1900,2000
language,,,,,,,,,
go,100.00,100.00,100.00,100.00,100.0,100.0,100.0,100.0,100.0
java,100.00,100.00,100.00,100.00,100.0,100.0,100.0,100.0,100.0
javascript,31.65,63.29,94.94,100.00,100.0,100.0,100.0,100.0,100.0
php,24.30,48.59,72.89,97.18,100.0,100.0,100.0,100.0,100.0
python,39.49,78.99,100.00,100.00,100.0,100.0,100.0,100.0,100.0
ruby,81.43,100.00,100.00,100.00,100.0,100.0,100.0,100.0,NaN



SAMPLE-SIZE DESIGN COMPARISON COMPLETED
No sample size or records were selected.


In [5]:
DESIGN_TARGETS = {
    "maximum_sampling_fraction_percent": 65.0,
    "maximum_margin_percent": 2.70,
    "maximum_rq1_paired_effect_mde": 0.10,
    "maximum_rq2_correlation_gap_mde": 0.17,
    "minimum_per_complexity_quartile": 300,
    "maximum_repository_cap": 10,
}

sample_size_decision = sample_size_comparison.copy()

sample_size_decision[
    "sampling_fraction_pass"
] = (
    sample_size_decision[
        "largest_sampling_fraction_percent"
    ]
    <= DESIGN_TARGETS[
        "maximum_sampling_fraction_percent"
    ]
)

sample_size_decision[
    "precision_pass"
] = (
    sample_size_decision[
        "worst_margin_percent"
    ]
    <= DESIGN_TARGETS[
        "maximum_margin_percent"
    ]
)

sample_size_decision[
    "rq1_power_pass"
] = (
    sample_size_decision[
        "rq1_paired_effect_mde_90"
    ]
    <= DESIGN_TARGETS[
        "maximum_rq1_paired_effect_mde"
    ]
)

sample_size_decision[
    "rq2_power_pass"
] = (
    sample_size_decision[
        "rq2_correlation_gap_mde_90"
    ]
    <= DESIGN_TARGETS[
        "maximum_rq2_correlation_gap_mde"
    ]
)

sample_size_decision[
    "complexity_subgroup_pass"
] = (
    sample_size_decision[
        "minimum_per_complexity_quartile"
    ]
    >= DESIGN_TARGETS[
        "minimum_per_complexity_quartile"
    ]
)

sample_size_decision[
    "repository_cap_pass"
] = (
    sample_size_decision[
        "largest_repository_cap_needed"
    ]
    <= DESIGN_TARGETS[
        "maximum_repository_cap"
    ]
)

criteria_columns = [
    "all_languages_feasible",
    "sampling_fraction_pass",
    "precision_pass",
    "rq1_power_pass",
    "rq2_power_pass",
    "complexity_subgroup_pass",
    "repository_cap_pass",
]

sample_size_decision[
    "all_design_criteria_pass"
] = (
    sample_size_decision[
        criteria_columns
    ].fillna(False).all(axis=1)
)

passing_candidates = (
    sample_size_decision.loc[
        sample_size_decision[
            "all_design_criteria_pass"
        ]
    ]
    .sort_values("sample_size_per_language")
)

assert not passing_candidates.empty, (
    "No candidate sample size satisfies all design targets."
)

selected_design = passing_candidates.iloc[0]

FINAL_SAMPLE_SIZE_PER_LANGUAGE = int(
    selected_design["sample_size_per_language"]
)

FINAL_BALANCED_SAMPLE_SIZE = (
    FINAL_SAMPLE_SIZE_PER_LANGUAGE
    * len(LANGUAGES)
)

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    with connection:
        connection.execute("""
            CREATE TABLE IF NOT EXISTS sample_design_metadata (
                key TEXT PRIMARY KEY,
                value TEXT NOT NULL
            )
        """)

        decision_metadata = {
            "selection_rule": (
                "smallest candidate satisfying all declared "
                "design criteria"
            ),
            "sample_size_per_language": str(
                FINAL_SAMPLE_SIZE_PER_LANGUAGE
            ),
            "balanced_total_sample": str(
                FINAL_BALANCED_SAMPLE_SIZE
            ),
            "statistical_power": str(POWER),
            "maximum_sampling_fraction_percent": str(
                DESIGN_TARGETS[
                    "maximum_sampling_fraction_percent"
                ]
            ),
            "maximum_margin_percent": str(
                DESIGN_TARGETS[
                    "maximum_margin_percent"
                ]
            ),
            "maximum_rq1_paired_effect_mde": str(
                DESIGN_TARGETS[
                    "maximum_rq1_paired_effect_mde"
                ]
            ),
            "maximum_rq2_correlation_gap_mde": str(
                DESIGN_TARGETS[
                    "maximum_rq2_correlation_gap_mde"
                ]
            ),
            "minimum_per_complexity_quartile": str(
                DESIGN_TARGETS[
                    "minimum_per_complexity_quartile"
                ]
            ),
            "maximum_repository_cap": str(
                DESIGN_TARGETS[
                    "maximum_repository_cap"
                ]
            ),
            "observed_worst_margin_percent": str(
                selected_design[
                    "worst_margin_percent"
                ]
            ),
            "observed_rq1_paired_effect_mde": str(
                selected_design[
                    "rq1_paired_effect_mde_90"
                ]
            ),
            "observed_rq2_correlation_gap_mde": str(
                selected_design[
                    "rq2_correlation_gap_mde_90"
                ]
            ),
            "observed_minimum_per_complexity_quartile": str(
                int(
                    selected_design[
                        "minimum_per_complexity_quartile"
                    ]
                )
            ),
            "observed_largest_repository_cap": str(
                int(
                    selected_design[
                        "largest_repository_cap_needed"
                    ]
                )
            ),
        }

        connection.executemany(
            """
            INSERT INTO sample_design_metadata(key, value)
            VALUES (?, ?)
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            decision_metadata.items(),
        )

finally:
    connection.close()


display_columns = [
    "sample_size_per_language",
    "balanced_total_sample",
    "all_languages_feasible",
    "largest_sampling_fraction_percent",
    "worst_margin_percent",
    "rq1_paired_effect_mde_90",
    "rq2_correlation_gap_mde_90",
    "minimum_per_complexity_quartile",
    "largest_repository_cap_needed",
    "all_design_criteria_pass",
]

print("Sample-size decision")
display(
    sample_size_decision[
        display_columns
    ]
)

print(
    "\nSelected sample size per language:",
    f"{FINAL_SAMPLE_SIZE_PER_LANGUAGE:,}",
)

print(
    "Total balanced sample:",
    f"{FINAL_BALANCED_SAMPLE_SIZE:,}",
)

assert FINAL_SAMPLE_SIZE_PER_LANGUAGE == int(
    passing_candidates[
        "sample_size_per_language"
    ].min()
)

print("\nSAMPLE SIZE SELECTED AND DOCUMENTED")
print("No sample records were selected yet.")

Sample-size decision


,sample_size_per_language,balanced_total_sample,all_languages_feasible,largest_sampling_fraction_percent,worst_margin_percent,rq1_paired_effect_mde_90,rq2_correlation_gap_mde_90,minimum_per_complexity_quartile,largest_repository_cap_needed,all_design_criteria_pass
0,250,1500.0,True,12.56,6.165,0.2228,0.3622,62.0,2.0,False
1,500,3000.0,True,25.13,4.335,0.1576,0.2613,125.0,3.0,False
2,750,4500.0,True,37.69,3.520,0.1286,0.2148,187.0,5.0,False
3,1000,6000.0,True,50.25,3.032,0.1114,0.1866,250.0,6.0,False
4,1250,7500.0,True,62.81,2.696,0.0996,0.1673,312.0,9.0,True
5,1500,9000.0,True,75.38,2.447,0.0910,0.1529,375.0,17.0,False
6,1750,10500.0,True,87.94,2.253,0.0842,0.1417,437.0,36.0,False
7,1900,11400.0,True,95.48,2.154,0.0808,0.1361,475.0,80.0,False
8,2000,NaN,False,39.53,2.095,0.0788,0.1327,NaN,16.0,False



Selected sample size per language: 1,250
Total balanced sample: 7,500

SAMPLE SIZE SELECTED AND DOCUMENTED
No sample records were selected yet.


In [6]:
import csv
import gzip
import hashlib
import json
import sqlite3
from collections import defaultdict
from datetime import datetime, timezone

SAMPLING_SEED = 20260805
OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SAMPLE_CSV_PATH = (
    OUTPUT_DIR
    / "01_5_Eligible_Test_Sample_7500.csv"
)

SAMPLE_METADATA_PATH = (
    OUTPUT_DIR
    / "01_5_Eligible_Test_Sample_7500_metadata.json"
)


def stable_uint64(*parts):
    value = "\x1f".join(
        str(part)
        for part in parts
    )

    digest = hashlib.sha256(
        value.encode("utf-8")
    ).digest()

    return int.from_bytes(
        digest[:8],
        byteorder="big",
        signed=False,
    )


connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    eligible_test_records = pd.read_sql_query("""
        SELECT
            family_id,
            language,
            repository,
            original_split,
            file_path,
            function_name,
            commit_sha,
            source_url,
            shard_relative_path,
            shard_line_number,
            code_token_count,
            comment_token_count,
            combined_token_count,
            code_line_count,
            comment_line_count,
            exact_code_hash,
            exact_comment_hash,
            exact_pair_hash,
            pair_token_count_capped
        FROM eligible_candidate_records
        WHERE original_split = 'test'
        ORDER BY language, repository, family_id
    """, connection)

finally:
    connection.close()


eligible_test_records[
    "repository_hash"
] = eligible_test_records.apply(
    lambda row: stable_uint64(
        SAMPLING_SEED,
        row["language"],
        row["repository"],
    ),
    axis=1,
)

eligible_test_records[
    "method_hash"
] = eligible_test_records["family_id"].map(
    lambda family_id: stable_uint64(
        SAMPLING_SEED,
        family_id,
    )
)

eligible_test_records = (
    eligible_test_records
    .sort_values(
        [
            "language",
            "repository",
            "method_hash",
            "family_id",
        ]
    )
    .reset_index(drop=True)
)

eligible_test_records[
    "within_repository_rank"
] = (
    eligible_test_records
    .groupby(
        [
            "language",
            "repository",
        ]
    )
    .cumcount()
    + 1
)


selected_language_frames = []
repository_cap_by_language = {}

for language in LANGUAGES:
    language_candidates = (
        eligible_test_records.loc[
            eligible_test_records[
                "language"
            ] == language
        ]
        .copy()
    )

    cap_row = sample_size_detail.loc[
        (
            sample_size_detail["language"]
            == language
        )
        & (
            sample_size_detail["sample_size"]
            == FINAL_SAMPLE_SIZE_PER_LANGUAGE
        ),
        "minimum_repository_cap_needed",
    ]

    assert len(cap_row) == 1

    repository_cap = int(
        cap_row.iloc[0]
    )

    repository_cap_by_language[
        language
    ] = repository_cap

    capped_candidates = (
        language_candidates.loc[
            language_candidates[
                "within_repository_rank"
            ] <= repository_cap
        ]
        .sort_values(
            [
                "within_repository_rank",
                "repository_hash",
                "method_hash",
                "family_id",
            ]
        )
        .head(
            FINAL_SAMPLE_SIZE_PER_LANGUAGE
        )
        .copy()
    )

    assert len(capped_candidates) == (
        FINAL_SAMPLE_SIZE_PER_LANGUAGE
    )

    capped_candidates[
        "sample_order"
    ] = range(
        1,
        FINAL_SAMPLE_SIZE_PER_LANGUAGE + 1,
    )

    capped_candidates[
        "repository_cap"
    ] = repository_cap

    selected_language_frames.append(
        capped_candidates
    )


selected_sample = (
    pd.concat(
        selected_language_frames,
        ignore_index=True,
    )
    .sort_values(
        [
            "language",
            "sample_order",
        ]
    )
    .reset_index(drop=True)
)


connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    with connection:
        connection.execute(
            "DROP TABLE IF EXISTS selected_test_sample"
        )

        connection.execute("""
            CREATE TABLE selected_test_sample (
                family_id TEXT PRIMARY KEY,
                language TEXT NOT NULL,
                repository TEXT NOT NULL,
                sample_order INTEGER NOT NULL,
                repository_round INTEGER NOT NULL,
                repository_cap INTEGER NOT NULL,
                selection_seed INTEGER NOT NULL,
                selection_method TEXT NOT NULL
            )
        """)

        insert_rows = [
            (
                row.family_id,
                row.language,
                row.repository,
                int(row.sample_order),
                int(row.within_repository_rank),
                int(row.repository_cap),
                SAMPLING_SEED,
                (
                    "deterministic repository-balanced "
                    "round-robin sampling"
                ),
            )
            for row in selected_sample.itertuples(
                index=False
            )
        ]

        connection.executemany(
            """
            INSERT INTO selected_test_sample (
                family_id,
                language,
                repository,
                sample_order,
                repository_round,
                repository_cap,
                selection_seed,
                selection_method
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?)
            """,
            insert_rows,
        )

        connection.execute("""
            CREATE INDEX
            idx_selected_test_sample_language
            ON selected_test_sample(
                language,
                sample_order
            )
        """)

        connection.execute(
            """
            INSERT INTO sample_design_metadata(
                key,
                value
            )
            VALUES (
                'sampling_seed',
                ?
            )
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            (str(SAMPLING_SEED),),
        )

        connection.execute(
            """
            INSERT INTO sample_design_metadata(
                key,
                value
            )
            VALUES (
                'selection_method',
                ?
            )
            ON CONFLICT(key)
            DO UPDATE SET value = excluded.value
            """,
            (
                "deterministic repository-balanced "
                "round-robin sampling",
            ),
        )

finally:
    connection.close()


targets_by_shard = defaultdict(dict)

for row in selected_sample.itertuples(
    index=False
):
    targets_by_shard[
        row.shard_relative_path
    ][int(row.shard_line_number)] = (
        row.family_id
    )


expected_hashes = {
    row.family_id: (
        row.exact_code_hash,
        row.exact_comment_hash,
    )
    for row in selected_sample.itertuples(
        index=False
    )
}

raw_text_by_family = {}

for shard_relative_path, line_targets in (
    targets_by_shard.items()
):
    shard_path = (
        PROJECT_ROOT
        / shard_relative_path
    )

    found_in_shard = 0

    with gzip.open(
        shard_path,
        mode="rt",
        encoding="utf-8",
        errors="strict",
    ) as handle:

        for line_number, raw_line in enumerate(
            handle,
            start=1,
        ):
            if line_number not in line_targets:
                continue

            record = json.loads(raw_line)
            family_id = line_targets[line_number]

            code = record["code"]
            comment = record["docstring"]

            code_hash = hashlib.sha256(
                code.encode("utf-8")
            ).hexdigest()

            comment_hash = hashlib.sha256(
                comment.encode("utf-8")
            ).hexdigest()

            assert (
                code_hash,
                comment_hash,
            ) == expected_hashes[family_id]

            raw_text_by_family[family_id] = (
                code,
                comment,
            )

            found_in_shard += 1

            if found_in_shard == len(
                line_targets
            ):
                break

    assert found_in_shard == len(
        line_targets
    )


selected_sample[
    "original_code"
] = selected_sample["family_id"].map(
    lambda family_id: (
        raw_text_by_family[family_id][0]
    )
)

selected_sample[
    "original_comment"
] = selected_sample["family_id"].map(
    lambda family_id: (
        raw_text_by_family[family_id][1]
    )
)

selected_sample[
    "sample_id"
] = selected_sample.apply(
    lambda row: (
        f"{row['language'].upper()}-"
        f"{int(row['sample_order']):04d}"
    ),
    axis=1,
)


export_columns = [
    "sample_id",
    "family_id",
    "language",
    "sample_order",
    "repository",
    "within_repository_rank",
    "repository_cap",
    "original_split",
    "file_path",
    "function_name",
    "commit_sha",
    "source_url",
    "shard_relative_path",
    "shard_line_number",
    "code_token_count",
    "comment_token_count",
    "combined_token_count",
    "pair_token_count_capped",
    "code_line_count",
    "comment_line_count",
    "exact_code_hash",
    "exact_comment_hash",
    "exact_pair_hash",
    "original_code",
    "original_comment",
]

sample_export = (
    selected_sample[
        export_columns
    ]
    .sort_values(
        [
            "language",
            "sample_order",
        ]
    )
    .reset_index(drop=True)
)

sample_export.to_csv(
    SAMPLE_CSV_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)


repository_validation = (
    sample_export
    .groupby(
        [
            "language",
            "repository",
        ]
    )
    .size()
    .rename("sampled_methods")
    .reset_index()
)

sample_validation = (
    sample_export
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        sampled_records=(
            "family_id",
            "count",
        ),
        sampled_repositories=(
            "repository",
            "nunique",
        ),
        maximum_methods_per_repository=(
            "within_repository_rank",
            "max",
        ),
        maximum_codebert_pair_tokens=(
            "pair_token_count_capped",
            "max",
        ),
    )
)

eligible_repository_counts = (
    eligible_test_records
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        eligible_test_repositories=(
            "repository",
            "nunique",
        )
    )
)

sample_validation = sample_validation.merge(
    eligible_repository_counts,
    on="language",
    how="left",
    validate="one_to_one",
)

sample_validation[
    "repository_coverage_percent"
] = (
    100
    * sample_validation[
        "sampled_repositories"
    ]
    / sample_validation[
        "eligible_test_repositories"
    ]
).round(2)

sample_validation[
    "declared_repository_cap"
] = (
    sample_validation["language"].map(
        repository_cap_by_language
    )
)


metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_database": str(
        CANDIDATE_DB_PATH
    ),
    "output_csv": str(
        SAMPLE_CSV_PATH
    ),
    "selection_seed": SAMPLING_SEED,
    "selection_method": (
        "deterministic repository-balanced "
        "round-robin sampling"
    ),
    "sample_size_per_language": (
        FINAL_SAMPLE_SIZE_PER_LANGUAGE
    ),
    "total_sample_size": (
        FINAL_BALANCED_SAMPLE_SIZE
    ),
    "partition": "test",
    "eligibility_rule": (
        "complete CodeBERT code-comment pair "
        "contains no more than 512 tokens"
    ),
    "codebert_checkpoint": (
        CODEBERT_CHECKPOINT
    ),
    "repository_cap_by_language": (
        repository_cap_by_language
    ),
    "design_targets": DESIGN_TARGETS,
}

with SAMPLE_METADATA_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        metadata,
        handle,
        indent=2,
        ensure_ascii=False,
    )


assert len(sample_export) == (
    FINAL_BALANCED_SAMPLE_SIZE
)

assert (
    sample_validation["sampled_records"]
    == FINAL_SAMPLE_SIZE_PER_LANGUAGE
).all()

assert (
    sample_export["original_split"]
    == "test"
).all()

assert (
    sample_export[
        "pair_token_count_capped"
    ] <= CODEBERT_MAX_LENGTH
).all()

assert not sample_export[
    "family_id"
].duplicated().any()

assert not sample_export[
    "exact_code_hash"
].duplicated().any()

assert not sample_export[
    "exact_pair_hash"
].duplicated().any()

assert (
    sample_validation[
        "sampled_repositories"
    ]
    == sample_validation[
        "eligible_test_repositories"
    ]
).all()

assert (
    sample_validation[
        "maximum_methods_per_repository"
    ]
    <= sample_validation[
        "declared_repository_cap"
    ]
).all()

assert sample_export[
    "original_code"
].notna().all()

assert sample_export[
    "original_comment"
].notna().all()


print("Final sample validation")
display(sample_validation)

print(
    "\nSample CSV:",
    SAMPLE_CSV_PATH,
)

print(
    "Metadata JSON:",
    SAMPLE_METADATA_PATH,
)

print(
    "CSV size:",
    f"{SAMPLE_CSV_PATH.stat().st_size / 1_000_000:.2f} MB",
)

print("\nFINAL SAMPLE SELECTED, VALIDATED, AND EXPORTED")

Final sample validation


,language,sampled_records,sampled_repositories,maximum_methods_per_repository,maximum_codebert_pair_tokens,eligible_test_repositories,repository_coverage_percent,declared_repository_cap
0,go,1250,211,8,508,211,100.0,8
1,java,1250,234,7,512,234,100.0,7
2,javascript,1250,790,2,512,790,100.0,2
3,php,1250,1029,2,512,1029,100.0,2
4,python,1250,633,3,511,633,100.0,3
5,ruby,1250,307,9,510,307,100.0,9



Sample CSV: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\01_5_Eligible_Test_Sample_7500.csv
Metadata JSON: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\01_5_Eligible_Test_Sample_7500_metadata.json
CSV size: 9.02 MB

FINAL SAMPLE SELECTED, VALIDATED, AND EXPORTED
